#Finalidad del Laboratorio
Crear un modelo que pueda identificar automáticamente el perfil de un pasajero de avión combinando dos características clave:
- el tipo de cliente que es (nuevo o frecuente),
- y la clase en la que viaja (económica, ejecutiva o business).
Con regresión logística multiclase

#¿Para qué sirve?
Este modelo puede servir para:
- entender mejor el comportamiento de los pasajeros,
- predecir qué tipo de cliente está viajando,
- y eventualmente mejorar decisiones comerciales o de servicio en aerolíneas, como personalización de atención, segmentación de marketing o análisis de satisfacción.



#Importar Librerias Necesarias

In [ ]:
# utilizado para manejos de directorios y rutas
import os

# Computacion vectorial y cientifica para python
import numpy as np

# Librerias para graficación (trazado de gráficos)
#from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Necesario para graficar superficies 3D

# Librería para manejo de datos
import pandas as pd

#Para separa el Dataset 20% y 80% para diferentes pruebas
from sklearn.model_selection import train_test_split

# llama a matplotlib a embeber graficas dentro de los cuadernillos
%matplotlib inline

 # 1. Exploración Inicial del Datase
#CARGA DE DATOS
Se carga el archivo test.csv en un DataFrame de pandas.

In [ ]:
# Cargar el conjunto de datos que convierte en un DataFrame de pandas
data = pd.read_csv('test.csv')
#data

#Deteccion de valores Nulos
Se contabilizan los valores nulos por columna. Esto permite decidir si se deben eliminar, imputar o tratar de otra forma antes del entrenamiento.

In [ ]:
# mostrar la informacion general del DataFrame
print('INFORMACIÓN DE TIPO DE DATOS')
data.info()

# muestra cuantos valores nulos (vacíos) hay en cada columna
print('\nDATOS VACÍOS')
print(pd.isnull(data).sum())

INFORMACIÓN DE TIPO DE DATOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25976 entries, 0 to 25975
Data columns (total 25 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Unnamed: 0                         25976 non-null  int64  
 1   id                                 25976 non-null  int64  
 2   Gender                             25976 non-null  object 
 3   Customer Type                      25976 non-null  object 
 4   Age                                25976 non-null  int64  
 5   Type of Travel                     25976 non-null  object 
 6   Class                              25976 non-null  object 
 7   Flight Distance                    25976 non-null  int64  
 8   Inflight wifi service              25976 non-null  int64  
 9   Departure/Arrival time convenient  25976 non-null  int64  
 10  Ease of Online booking             25976 non-null  int64  
 11  Gate location            

# 2. Selección Explícita de Variables Relevante
#Extracción Manual de Columna
Se realiza una selección explícita de variables predictoras y de la variable objetivo (satisfaction).


In [ ]:
# Selección explícita de columnas relevantes
data = data[[
    'Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class', 'Flight Distance',
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service',
    'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes',
    'satisfaction'
]]


mostrar los datos del DataFrame

In [ ]:
data

,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Female,Loyal Customer,52,Business travel,Eco,160,5,4,3,4,...,5,5,5,5,2,5,5,50,44.0,satisfied
1,Female,Loyal Customer,36,Business travel,Business,2863,1,1,3,1,...,4,4,4,4,3,4,5,0,0.0,satisfied
2,Male,disloyal Customer,20,Business travel,Eco,192,2,0,2,4,...,2,4,1,3,2,2,2,0,0.0,neutral or dissatisfied
3,Male,Loyal Customer,44,Business travel,Business,3377,0,0,0,2,...,1,1,1,1,3,1,4,0,6.0,satisfied
4,Female,Loyal Customer,49,Business travel,Eco,1182,2,3,4,3,...,2,2,2,2,4,2,4,0,20.0,satisfied
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25971,Male,disloyal Customer,34,Business travel,Business,526,3,3,3,1,...,4,3,2,4,4,5,4,0,0.0,neutral or dissatisfied
25972,Male,Loyal Customer,23,Business travel,Business,646,4,4,4,4,...,4,4,5,5,5,5,4,0,0.0,satisfied
25973,Female,Loyal Customer,17,Personal Travel,Eco,828,2,5,1,5,...,2,4,3,4,5,4,2,0,0.0,neutral or dissatisfied
25974,Male,Loyal Customer,14,Business travel,Business,1127,3,3,3,3,...,4,3,2,5,4,5,4,0,0.0,satisfied


# 3. Tipos de Datos y Valores Único
#Verificación de Tipos de Datos
muestra el tipo de dato de cada columna del DataFrame.

In [ ]:
# muestra los tipos de datos de cada columna y los valores únicos en la columna 'satisfaction'
print(data.dtypes)
print(data['satisfaction'].unique())

Gender                                object
Customer Type                         object
Age                                    int64
Type of Travel                        object
Class                                 object
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Inflight service                       int64
Cleanliness                            int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
satisfacti

# 4. Codificación de Variables Categóricas
#Aplicación de LabelEncoder
Se utiliza  para transformar variables categóricas en valores numéricos. Esto es esencial para que los algoritmos de aprendizaje automático puedan procesar los datos sin ambigüedad.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# LabelEncoder convierte valores de texto (como 'Male', 'Female', 'Business', etc.) en números (0, 1, 2, ...).
# El bucle aplica esta conversión a cada columna categórica de la lista,
# reemplazando los textos por números en el DataFrame.

# Codifica variables categóricas
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']:
    data[col] = LabelEncoder().fit_transform(data[col])

# Verifica los tipos de datos y valores únicos de la columna objetivo
print(data.dtypes)
print(data['satisfaction'].unique())

Gender                                 int64
Customer Type                          int64
Age                                    int64
Type of Travel                         int64
Class                                  int64
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Inflight service                       int64
Cleanliness                            int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
satisfacti

muestra los datos una vez transformados las variables

In [ ]:
data

,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,0,52,0,1,160,5,4,3,4,...,5,5,5,5,2,5,5,50,44.0,1
1,0,0,36,0,0,2863,1,1,3,1,...,4,4,4,4,3,4,5,0,0.0,1
2,1,1,20,0,1,192,2,0,2,4,...,2,4,1,3,2,2,2,0,0.0,0
3,1,0,44,0,0,3377,0,0,0,2,...,1,1,1,1,3,1,4,0,6.0,1
4,0,0,49,0,1,1182,2,3,4,3,...,2,2,2,2,4,2,4,0,20.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25971,1,1,34,0,0,526,3,3,3,1,...,4,3,2,4,4,5,4,0,0.0,0
25972,1,0,23,0,0,646,4,4,4,4,...,4,4,5,5,5,5,4,0,0.0,1
25973,0,0,17,1,1,828,2,5,1,5,...,2,4,3,4,5,4,2,0,0.0,0
25974,1,0,14,0,0,1127,3,3,3,3,...,4,3,2,5,4,5,4,0,0.0,1


# 5. Creación de Variable Multiclase
#Combinación de Variables Categóricas
Se genera una nueva columna multiclase que concatena los valores codificados de Customer Type y Class. Esto permite representar combinaciones únicas como "0-1", "1-2", etc., que reflejan el perfil del pasajero según su tipo y clase de vuelo


Se aplica LabelEncoder para transformar cada combinación textual en un valor numérico único. Esto convierte la variable multiclase en una etiqueta multiclase lista para ser utilizada como variable objetivo en el modelo de clasificación.


In [ ]:
# Se crea una nueva columna llamada multiclase que combina los valores de 'Customer Type' y 'Class' (por ejemplo, "0-1").
# Luego, esa columna se codifica a números con LabelEncoder, para que cada combinación tenga un valor único (0, 1, 2, ...).
# Se imprime qué valores únicos tiene la columna multiclase y un ejemplo de cómo quedaron las primeras fila

# Crear columna multiclase combinando Customer Type y Class
data['multiclase'] = data['Customer Type'].astype(str) + '-' + data['Class'].astype(str)

# Codificar la nueva columna multiclase
from sklearn.preprocessing import LabelEncoder
data['multiclase'] = LabelEncoder().fit_transform(data['multiclase'])

# Mostrar las clases creadas
print("Clases multiclase:", data['multiclase'].unique())
print(data[['Customer Type', 'Class', 'multiclase']].head())

Clases multiclase: [1 0 4 2 3 5]
   Customer Type  Class  multiclase
0              0      1           1
1              0      0           0
2              1      1           4
3              0      0           0
4              0      1           1


muestra los datos con la columna multiclase

In [ ]:
data

,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction,multiclase
0,0,0,52,0,1,160,5,4,3,4,...,5,5,5,2,5,5,50,44.0,1,1
1,0,0,36,0,0,2863,1,1,3,1,...,4,4,4,3,4,5,0,0.0,1,0
2,1,1,20,0,1,192,2,0,2,4,...,4,1,3,2,2,2,0,0.0,0,4
3,1,0,44,0,0,3377,0,0,0,2,...,1,1,1,3,1,4,0,6.0,1,0
4,0,0,49,0,1,1182,2,3,4,3,...,2,2,2,4,2,4,0,20.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25971,1,1,34,0,0,526,3,3,3,1,...,3,2,4,4,5,4,0,0.0,0,3
25972,1,0,23,0,0,646,4,4,4,4,...,4,5,5,5,5,4,0,0.0,1,0
25973,0,0,17,1,1,828,2,5,1,5,...,4,3,4,5,4,2,0,0.0,0,1
25974,1,0,14,0,0,1127,3,3,3,3,...,3,2,5,4,5,4,0,0.0,1,0


In [ ]:
# El balanceo de la columna multiclase para ver cuántos ejemplos hay de cada clase.

#import matplotlib.pyplot as plt

#plt.figure(figsize=(8,4))
#data['multiclase'].value_counts().sort_index().plot(kind='bar')
#plt.xlabel('Clase multiclase')
#plt.ylabel('Cantidad')
#plt.title('Distribución de clases multiclase')
#plt.show()

# 6. Verificación de Valores Nulos
#Conteo de Valores Vacíos por Columna

In [ ]:
# Muestra cuántos valores nulos (vacíos) hay en cada columna del DataFrame.
# Verificar si hay valores nulos en el DataFrame
print("Valores nulos por columna:")
print(data.isnull().sum())

Valores nulos por columna:
Gender                                0
Customer Type                         0
Age                                   0
Type of Travel                        0
Class                                 0
Flight Distance                       0
Inflight wifi service                 0
Departure/Arrival time convenient     0
Ease of Online booking                0
Gate location                         0
Food and drink                        0
Online boarding                       0
Seat comfort                          0
Inflight entertainment                0
On-board service                      0
Leg room service                      0
Baggage handling                      0
Checkin service                       0
Inflight service                      0
Cleanliness                           0
Departure Delay in Minutes            0
Arrival Delay in Minutes             83
satisfaction                          0
multiclase                            0
dtype: int64


# 7. Imputación de Valores Nulos
#Relleno de Valores Faltantes en Arrival Delay in Minutes
Se identifican los valores nulos en la columna Arrival Delay in Minutes y se reemplazan por el promedio de la misma. Esta técnica de imputación mantiene la coherencia estadística del conjunto de datos sin introducir sesgos extremos.


In [ ]:
# Busca los valores nulos (NaN) en la columna 'Arrival Delay in Minutes'.
# Los reemplaza por el promedio de esa columna.

# Rellenar los nulos de 'Arrival Delay in Minutes' con el promedio de la columna
data['Arrival Delay in Minutes'].fillna(data['Arrival Delay in Minutes'].mean(), inplace=True)

C:\Users\Infosat\AppData\Local\Temp\ipykernel_4372\1025543996.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Arrival Delay in Minutes'].fillna(data['Arrival Delay in Minutes'].mean(), inplace=True)


#Conteo de Valores Vacíos por Columna

In [ ]:
# Verificar si hay valores nulos en el DataFrame
print("Valores nulos por columna:")
print(data.isnull().sum())

Valores nulos por columna:
Gender                               0
Customer Type                        0
Age                                  0
Type of Travel                       0
Class                                0
Flight Distance                      0
Inflight wifi service                0
Departure/Arrival time convenient    0
Ease of Online booking               0
Gate location                        0
Food and drink                       0
Online boarding                      0
Seat comfort                         0
Inflight entertainment               0
On-board service                     0
Leg room service                     0
Baggage handling                     0
Checkin service                      0
Inflight service                     0
Cleanliness                          0
Departure Delay in Minutes           0
Arrival Delay in Minutes             0
satisfaction                         0
multiclase                           0
dtype: int64


# 8. Separación de Variables Predictoras y Objetivo
#División Manual de X e y
Se seleccionan explícitamente las columnas que serán utilizadas como variables predictoras (X) y se asigna la variable objetivo (y) como la columna multiclase.


In [ ]:
# Separar X y y seleccionando explícitamente las columnas (como en el ejemplo)
X = data[['Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class', 'Flight Distance',
          'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
          'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment',
          'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service',
          'Inflight service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']]

y = data['multiclase']

#Imprimir X

In [ ]:
print(X)

       Gender  Customer Type  Age  Type of Travel  Class  Flight Distance  \
0           0              0   52               0      1              160   
1           0              0   36               0      0             2863   
2           1              1   20               0      1              192   
3           1              0   44               0      0             3377   
4           0              0   49               0      1             1182   
...       ...            ...  ...             ...    ...              ...   
25971       1              1   34               0      0              526   
25972       1              0   23               0      0              646   
25973       0              0   17               1      1              828   
25974       1              0   14               0      0             1127   
25975       0              0   42               1      1              264   

       Inflight wifi service  Departure/Arrival time convenient  \
0       

#Imprimir y

In [ ]:
print(y)

0        1
1        0
2        4
3        0
4        1
        ..
25971    3
25972    0
25973    1
25974    0
25975    1
Name: multiclase, Length: 25976, dtype: int64


# 9. División del Dataset en Entrenamiento y Prueba
#Separación Estratificada con train_test_split
Se utiliza train_test_split para dividir el conjunto de datos en dos partes:
- 80 % para entrenamiento (X_train, y_train)
- 20 % para prueba (X_test, y_test)
La opción stratify=y garantiza que la distribución de clases en y se mantenga proporcional en ambos subconjuntos.

In [ ]:
# 1. Dividir el dataset en entrenamiento y prueba
# Se utiliza train_test_split para obtener conjuntos de entrenamiento y prueba (por ejemplo, 80% train, 20% test).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#Imprimir X_train de entrenamiento y X_test de prueba

In [ ]:
print(X_train)
print(X_test)

       Gender  Customer Type  Age  Type of Travel  Class  Flight Distance  \
23433       0              0   21               0      0              936   
9616        0              1   28               0      1             1506   
15928       0              0   47               0      0              680   
2364        0              0   19               1      1              967   
7622        1              0   29               1      1              853   
...       ...            ...  ...             ...    ...              ...   
13357       1              0   26               0      0              817   
8458        1              0   62               1      0              462   
18360       0              0   43               0      0             3642   
25615       0              0    9               1      1              370   
2627        0              0   62               1      1              170   

       Inflight wifi service  Departure/Arrival time convenient  \
23433   

#Imprimir y_train de entrenamiento y y_test de prueba

In [ ]:
print(y_train)
print(y_test)

23433    0
9616     4
15928    0
2364     1
7622     1
        ..
13357    0
8458     0
18360    0
25615    1
2627     1
Name: multiclase, Length: 20780, dtype: int64
16496    0
3643     2
4081     2
24362    0
21517    4
        ..
3499     1
9136     1
20636    0
22597    0
14228    4
Name: multiclase, Length: 5196, dtype: int64


# 10. Normalización de Datos de Entrenamiento
#Función Manual para Normalización
Se define una función personalizada  que aplica normalización estándar (media 0, desviación estándar 1) sobre cada columna de . Este proceso transforma las variables numéricas para que tengan la misma escala

In [ ]:
#2. Normalizar los datos de entrenamiento
#Se normalizan los datos de entrenamiento (media 0, varianza 1) usando una función manual o similar a la del ejemplo:

def featureNormalize(X):
    X_norm = X.copy()
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1  # Evitar división por cero
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_train_norm, mu, sigma = featureNormalize(X_train)

Normalizar los datos de prueba usando mu y sigma de entrenamiento

In [ ]:
#3. Normalizar los datos de prueba usando mu y sigma de entrenamiento

X_test_norm = (X_test - mu) / sigma

Mostrar dimensiones y primeros datos normalizados

In [ ]:
# 4. Mostrar dimensiones y primeros datos normalizados (opcional)

print("X_train_norm shape:", X_train_norm.shape)
print("X_test_norm shape:", X_test_norm.shape)
#print(X_train_norm[:5])

X_train_norm shape: (20780, 22)
X_test_norm shape: (5196, 22)


# 11. Inclusión del Término de Sesgo (Bias)
#Agregado de Columna de Unos a los Datos Normalizados
Se agrega una columna de unos al inicio de los conjuntos X_train_norm y X_test_norm. Esta columna representa el término de sesgo (bias) en la regresión logística, permitiendo que el modelo aprenda un desplazamiento constante en la función de decisión.

In [ ]:
# 5. Agregar una columna de unos a la izquierda de los datos normalizados
m_train = X_train_norm.shape[0]
m_test = X_test_norm.shape[0]

X_train_ready = np.concatenate([np.ones((m_train, 1)), X_train_norm], axis=1)
X_test_ready = np.concatenate([np.ones((m_test, 1)), X_test_norm], axis=1)

print("X_train_ready shape:", X_train_ready.shape)
print("X_test_ready shape:", X_test_ready.shape)

X_train_ready shape: (20780, 23)
X_test_ready shape: (5196, 23)


#Imprimir X_test_ready X de pruba listo

In [ ]:
print(X_test_ready)

[[ 1.          1.01522659 -0.47603571 ...  0.53762169  2.07882249
   1.8581758 ]
 [ 1.         -0.98500178 -0.47603571 ...  1.29724491 -0.38949595
  -0.4014256 ]
 [ 1.         -0.98500178 -0.47603571 ...  0.53762169  0.07674198
  -0.15342057]
 ...
 [ 1.          1.01522659 -0.47603571 ...  0.53762169 -0.06038683
  -0.18097668]
 [ 1.          1.01522659 -0.47603571 ...  1.29724491 -0.03296107
  -0.07075222]
 [ 1.          1.01522659  2.10068273 ...  1.29724491  2.70961498
   2.38174198]]


#Imprimir X_train_ready X de entrenamiento listo

In [ ]:
print(X_train_ready)

[[ 1.         -0.98500178 -0.47603571 ...  1.29724491 -0.38949595
  -0.4014256 ]
 [ 1.         -0.98500178  2.10068273 ... -1.74124797  0.21387078
  -0.31875726]
 [ 1.         -0.98500178 -0.47603571 ... -1.74124797  0.62525719
   0.48037007]
 ...
 [ 1.         -0.98500178 -0.47603571 ...  1.29724491 -0.38949595
  -0.4014256 ]
 [ 1.         -0.98500178 -0.47603571 ... -0.22200153 -0.38949595
  -0.31875726]
 [ 1.         -0.98500178 -0.47603571 ... -1.74124797 -0.38949595
  -0.4014256 ]]


#12. Definición de la Función Sigmoidea
#Implementación de sigmoid(z)
transforma cualquier valor real en un rango entre 0 y 1. Es utilizada como función de activación en regresión logística para interpretar la salida como una probabilidad.

In [ ]:
# Definir la función sigmoidea (sigmoid)
def sigmoid(z):
    """
    Calcula la sigmoide de z.
    """
    return 1.0 / (1.0 + np.exp(-z))

# Prueba la función sigmoidea con un valor
print("Sigmoid(0) =", sigmoid(0))  # Debería imprimir 0.5

Sigmoid(0) = 0.5


# 13. Función de Costo y Gradiente Regularizado
#Definición de lrCostFunction
Esta función calcula dos elementos clave para la regresión logística:
- J: el valor del costo regularizado, que mide qué tan bien se ajusta el modelo a los datos.
- grad: el gradiente, que indica cómo deben ajustarse los parámetros theta para minimizar el costo.
Se incluye regularización L2 para evitar sobreajuste, pero se excluye el término de sesgo (theta[0]) de la penalización, respetando la lógica matemática del modelo.
El uso de np.clip con epsilon evita errores numéricos al calcular logaritmos de valores cercanos a 0 o 1.


In [ ]:
# Función de costo y gradiente para regresión logística regularizada
def lrCostFunction(theta, X, y, lambda_):
    """
    Calcula el costo y el gradiente para regresión logística regularizada.
    """
    m = y.size
    h = sigmoid(X.dot(theta.T))
    epsilon = 1e-5  # Para evitar log(0)
    h = np.clip(h, epsilon, 1 - epsilon)
    temp = theta.copy()
    temp[0] = 0  # No regularizar el sesgo

    J = (1 / m) * np.sum(-y.dot(np.log(h)) - (1 - y).dot(np.log(1 - h))) + (lambda_ / (2 * m)) * np.sum(np.square(temp))
    grad = (1 / m) * (h - y).dot(X) + (lambda_ / m) * temp

    return J, grad

# 14. Entrenamiento Multiclase con One-vs-All
#Definición de la Función oneVsAll
La función oneVsAll entrena un clasificador binario para cada clase multiclase. Para cada iteración:
- Se crea un vector binario y_c que marca la clase actual como 1 y las demás como 0.
- Se inicializa theta en ceros.
- Se optimiza la función de costo regularizada usando el método TNC con un máximo de 50 iteraciones.
- Los parámetros optimizados se almacenan en all_theta, donde cada fila corresponde a una clase.

In [ ]:
# Ahora sigue la función One-vs-All para entrenar un clasificador por cada clase (multiclase), igual que en el ejemplo tradicional:
from scipy.optimize import minimize

def oneVsAll(X, y, num_labels, lambda_):
    """
    Entrena múltiples clasificadores de regresión logística (uno por clase).
    """
    m, n = X.shape
    all_theta = np.zeros((num_labels, n))

    for c in range(num_labels):
        # Crear vector binario para la clase actual
        y_c = (y == c).astype(int)
        initial_theta = np.zeros(n)

        # Minimizar la función de costo
        res = minimize(fun=lambda t: lrCostFunction(t, X, y_c, lambda_)[0],
                       x0=initial_theta,
                       jac=lambda t: lrCostFunction(t, X, y_c, lambda_)[1],
                       method='TNC',
                       options={'maxiter': 50})
        all_theta[c] = res.x
    return all_theta

# Número de clases
num_labels = len(np.unique(y_train))
lambda_ = 1

# Entrenar clasificadores One-vs-All
all_theta = oneVsAll(X_train_ready, y_train, num_labels, lambda_)

C:\Users\Infosat\AppData\Local\Temp\ipykernel_4372\2836050515.py:17: OptimizeWarning: Unknown solver options: maxiter
  res = minimize(fun=lambda t: lrCostFunction(t, X, y_c, lambda_)[0],
C:\Users\Infosat\AppData\Local\Temp\ipykernel_4372\2836050515.py:17: OptimizeWarning: Unknown solver options: maxiter
  res = minimize(fun=lambda t: lrCostFunction(t, X, y_c, lambda_)[0],
C:\Users\Infosat\AppData\Local\Temp\ipykernel_4372\2836050515.py:17: OptimizeWarning: Unknown solver options: maxiter
  res = minimize(fun=lambda t: lrCostFunction(t, X, y_c, lambda_)[0],
C:\Users\Infosat\AppData\Local\Temp\ipykernel_4372\2836050515.py:17: OptimizeWarning: Unknown solver options: maxiter
  res = minimize(fun=lambda t: lrCostFunction(t, X, y_c, lambda_)[0],
C:\Users\Infosat\AppData\Local\Temp\ipykernel_4372\2836050515.py:17: OptimizeWarning: Unknown solver options: maxiter
  res = minimize(fun=lambda t: lrCostFunction(t, X, y_c, lambda_)[0],
C:\Users\Infosat\AppData\Local\Temp\ipykernel_4372\283605051

# 15. Función oneVsAll: Entrenamiento Multiclase
#Implementación del Enfoque One-vs-All
Esta función entrena un clasificador binario para cada clase multiclase. El procedimiento es el siguiente:
- Se recorre cada clase c en el rango num_labels.
- Se genera un vector binario y_c donde los ejemplos de la clase actual se marcan como 1 y el resto como 0.
- Se inicializa el vector de parámetros theta en ceros.
- Se utiliza scipy.optimize.minimize para minimizar la función de costo regularizada lrCostFunction, con el método TNC y derivadas proporcionadas por jac.
- Los parámetros optimizados para cada clase se almacenan en la matriz all_theta, donde cada fila representa un clasificador independiente.

In [ ]:
from scipy.optimize import minimize

def oneVsAll(X, y, num_labels, lambda_):
    m, n = X.shape
    all_theta = np.zeros((num_labels, n))

    for c in range(num_labels):
        y_c = (y == c).astype(int)
        initial_theta = np.zeros(n)

        res = minimize(fun=lambda t: lrCostFunction(t, X, y_c, lambda_)[0],
                       x0=initial_theta,
                       jac=lambda t: lrCostFunction(t, X, y_c, lambda_)[1],
                       method='TNC',
                       options={'maxiter': 50})

        all_theta[c] = res.x

    return all_theta

# 16. Predicción Multiclase con One-vs-All
#Definición de la Función predictOneVsAll
La función predictOneVsAll calcula la probabilidad de pertenencia a cada clase utilizando los parámetros entrenados en all_theta. Para cada ejemplo:
- Se aplica la función sigmoidea sobre el producto matricial entre los datos y los parámetros.
- Se selecciona la clase con mayor probabilidad usando np.argmax.
La precisión se calcula como el porcentaje de ejemplos correctamente clasificados en el conjunto de prueba.


In [ ]:
# Ahora sigue la función de predicción para multiclase (One-vs-All), que predice la clase para cada ejemplo usando los clasificadores entrenados:

def predictOneVsAll(all_theta, X):
    """
    Predice la clase para cada ejemplo en X usando los clasificadores One-vs-All.
    """
    probs = sigmoid(X.dot(all_theta.T))  # Probabilidades para cada clase
    return np.argmax(probs, axis=1)      # Clase con mayor probabilidad

# Predecir en el conjunto de prueba
y_pred = predictOneVsAll(all_theta, X_test_ready)

# Calcular precisión
accuracy = np.mean(y_pred == y_test)
print(f"Precisión en el conjunto de prueba: {accuracy * 100:.2f}%")

Precisión en el conjunto de prueba: 99.23%


#Conclusión
La regresión logística multiclase logra una precisión de 99.23% sobre el conjunto de prueba. Aunque el modelo es interpretable y trazable, su rendimiento puede verse limitado por la linealidad de la función de decisión.
